# 🐍 Python Course — Object-Oriented Programming
## Notebook 2: Inheritance, Polymorphism, Abstract Classes & Dunder Methods

---

### 📚 What You Will Learn

| # | Topic | Description |
|---|---|---|
| 1 | Inheritance | Creating child classes from parent classes |
| 2 | `super()` | Calling parent class methods and constructors |
| 3 | Method Overriding | Redefining behavior in child classes |
| 4 | `isinstance` & `issubclass` | Checking object and class relationships |
| 5 | Multiple Inheritance | Inheriting from more than one parent |
| 6 | MRO — Method Resolution Order | How Python resolves method lookup |
| 7 | Polymorphism | One interface, many forms |
| 8 | Duck Typing | Python's flexible approach to types |
| 9 | Abstract Classes | Enforcing method implementation in subclasses |
| 10 | Dunder Methods | `__len__`, `__eq__`, `__add__`, `__iter__` and more |

> **Prerequisites:** Complete **OOP Part 1** (Classes, Objects, `__init__`, Encapsulation, Properties).

---

## 1. 🧬 Inheritance

**Inheritance** allows a class (child) to **reuse and extend** the attributes and methods of another class (parent).

- Avoids code duplication — write once, reuse many times.
- Models real-world "is-a" relationships: `Dog` **is-a** `Animal`, `Car` **is-a** `Vehicle`.

```
        Animal  (Parent / Base / Superclass)
           │
    ┌──────┴──────┐
   Dog           Cat     (Children / Derived / Subclasses)
    │
 GoldenRetriever          (Grandchild)
```

### Syntax
```python
class Child(Parent):
    ...
```

In [ ]:
# ── Parent Class ──────────────────────────────────────────
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age  = age

    def eat(self):
        print(f"{self.name} is eating.")

    def sleep(self):
        print(f"{self.name} is sleeping.")

    def __str__(self):
        return f"{self.__class__.__name__}(name={self.name}, age={self.age})"


# ── Child Class ───────────────────────────────────────────
class Dog(Animal):              # Dog inherits from Animal
    def bark(self):             # new method — only on Dog
        print(f"{self.name} says: Woof!")


class Cat(Animal):              # Cat also inherits from Animal
    def meow(self):
        print(f"{self.name} says: Meow!")


# Creating instances
dog = Dog("Rex", 3)
cat = Cat("Whiskers", 5)

# Inherited methods work on child objects
dog.eat()     # from Animal
dog.sleep()   # from Animal
dog.bark()    # Dog's own method

cat.eat()     # from Animal
cat.meow()    # Cat's own method

print(dog)    # uses Animal's __str__
print(cat)

---
## 2. 🔗 `super()` — Calling the Parent

`super()` gives you access to the **parent class** from within the child class. Most commonly used to:
1. Call the parent's `__init__` so you don't re-write the same setup code.
2. Call a parent method that you've overridden, but still want to execute.

In [ ]:
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age  = age
        print(f"Animal.__init__ called for {name}")

    def describe(self):
        print(f"I am {self.name}, aged {self.age}.")


class Dog(Animal):
    def __init__(self, name, age, breed):
        super().__init__(name, age)   # ← calls Animal.__init__
        self.breed = breed            # Dog's extra attribute
        print(f"Dog.__init__ called — breed: {breed}")

    def describe(self):
        super().describe()            # ← calls Animal.describe first
        print(f"I am a {self.breed}.") # then adds Dog-specific info


class GoldenRetriever(Dog):
    def __init__(self, name, age):
        super().__init__(name, age, "Golden Retriever")  # calls Dog.__init__
        print("GoldenRetriever.__init__ called")


print("─── Creating GoldenRetriever ───")
gr = GoldenRetriever("Buddy", 2)
print()
gr.describe()

---
## 3. 🔄 Method Overriding

A child class can **redefine** a method inherited from the parent. The child's version **replaces** the parent's when called on a child object.

```
  Animal.speak()  →  "..."      (generic)
  Dog.speak()     →  "Woof!"    (overridden)
  Cat.speak()     →  "Meow!"    (overridden)
```

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        print(f"{self.name} makes a sound.")

    def move(self):
        print(f"{self.name} moves.")


class Dog(Animal):
    def speak(self):           # overrides Animal.speak
        print(f"{self.name} says: Woof!")


class Cat(Animal):
    def speak(self):           # overrides Animal.speak
        print(f"{self.name} says: Meow!")

    def move(self):            # overrides Animal.move
        print(f"{self.name} sneaks silently.")


class Fish(Animal):
    pass                       # does NOT override speak — uses Animal's


animals = [
    Dog("Rex"),
    Cat("Whiskers"),
    Fish("Nemo"),
    Animal("Unknown")
]

print("── speak() ──")
for a in animals:
    a.speak()    # each calls its own version (or parent's if not overridden)

print("\n── move() ──")
for a in animals:
    a.move()

---
## 4. 🔍 `isinstance()` and `issubclass()`

These built-in functions let you check type relationships at runtime.

| Function | Returns `True` when |
|---|---|
| `isinstance(obj, Class)` | `obj` is an instance of `Class` **or any subclass** |
| `issubclass(Child, Parent)` | `Child` is a subclass of `Parent` |

In [ ]:
class Animal: pass
class Dog(Animal): pass
class GoldenRetriever(Dog): pass

gr = GoldenRetriever()

# isinstance checks the inheritance chain
print(isinstance(gr, GoldenRetriever))  # True  — direct type
print(isinstance(gr, Dog))              # True  — parent
print(isinstance(gr, Animal))           # True  — grandparent
print(isinstance(gr, list))             # False — unrelated type

print()
# issubclass checks class relationships
print(issubclass(GoldenRetriever, Dog))     # True
print(issubclass(GoldenRetriever, Animal))  # True
print(issubclass(Dog, GoldenRetriever))     # False — wrong direction
print(issubclass(Animal, Animal))           # True  — a class is a subclass of itself

print()
# Practical use: safe type checking before calling a method
animals = [Dog(), GoldenRetriever(), Animal(), "not an animal"]
for obj in animals:
    if isinstance(obj, Animal):
        print(f"{obj.__class__.__name__} is an Animal ✓")
    else:
        print(f"{obj!r} is NOT an Animal ✗")

---
## 5. 🧩 Multiple Inheritance

Python allows a class to inherit from **more than one parent**.

```python
class Child(Parent1, Parent2):
    ...
```

This is powerful but requires care — especially when both parents define the same method.

In [ ]:
class Flyable:
    def fly(self):
        print(f"{self.name} is flying.")

    def move(self):
        print(f"{self.name} moves by flying.")


class Swimmable:
    def swim(self):
        print(f"{self.name} is swimming.")

    def move(self):
        print(f"{self.name} moves by swimming.")


class Animal:
    def __init__(self, name):
        self.name = name


# Duck inherits from Animal, Flyable, AND Swimmable
class Duck(Animal, Flyable, Swimmable):
    def quack(self):
        print(f"{self.name} says: Quack!")


duck = Duck("Donald")
duck.fly()      # from Flyable
duck.swim()     # from Swimmable
duck.quack()    # Duck's own

# When both parents have move() — which one wins?
duck.move()     # Flyable.move() wins — see MRO section below

---
## 6. 🗺️ MRO — Method Resolution Order

When multiple parents have the same method, Python uses the **MRO** (C3 linearization algorithm) to decide which one to call.

**Rule of thumb:** Python searches **left to right, depth first**, but never visits a class before all its children have been visited.

You can inspect the MRO with `Class.__mro__` or `Class.mro()`.

In [ ]:
# Inspect the MRO of Duck
print(Duck.__mro__)
print()
# Printed as a readable list:
for i, cls in enumerate(Duck.__mro__):
    print(f"  {i+1}. {cls.__name__}")

In [ ]:
# Practical MRO example with a diamond problem
#
#        Base
#       /    \
#      A      B
#       \    /
#         C

class Base:
    def hello(self):
        print("Hello from Base")

class A(Base):
    def hello(self):
        print("Hello from A")
        super().hello()

class B(Base):
    def hello(self):
        print("Hello from B")
        super().hello()

class C(A, B):         # inherits from both A and B
    def hello(self):
        print("Hello from C")
        super().hello()


c = C()
c.hello()
print()
print("MRO:", [cls.__name__ for cls in C.__mro__])
# Python ensures Base.hello is called ONLY ONCE despite A and B both calling super()

---
## 7. 🎭 Polymorphism

**Polymorphism** means "many forms". In OOP it means: **the same interface (method name) can behave differently depending on the object**.

You've already seen this with method overriding. Now let's understand it more deeply.

```
  shape.area()  →  works on Circle, Rectangle, Triangle...
  Each shape calculates area differently — same call, different behavior.
```

In [ ]:
import math

class Shape:
    def area(self):
        return 0

    def perimeter(self):
        return 0

    def describe(self):
        print(f"{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}")


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width  = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


class Triangle(Shape):
    def __init__(self, a, b, c):
        self.a, self.b, self.c = a, b, c

    def area(self):
        s = self.perimeter() / 2          # semi-perimeter
        return math.sqrt(s * (s-self.a) * (s-self.b) * (s-self.c))

    def perimeter(self):
        return self.a + self.b + self.c


# ── Polymorphism in action ───────────────────────────────
shapes = [
    Circle(5),
    Rectangle(4, 6),
    Triangle(3, 4, 5),
]

# Same call — different behavior per object type
for shape in shapes:
    shape.describe()

print()
# Works with any function that accepts a Shape
def print_area(shape):
    print(f"Area of {shape.__class__.__name__}: {shape.area():.4f}")

for shape in shapes:
    print_area(shape)    # works for Circle, Rectangle, Triangle without any if/elif

---
## 8. 🦆 Duck Typing

> *"If it walks like a duck and quacks like a duck, it's a duck."*

Python doesn't require objects to inherit from a common class to be used interchangeably. If an object **has the right method**, Python will call it — no questions asked about type.

This is a key difference from strictly typed OOP languages like Java.

In [ ]:
# These classes share NO common parent — yet work polymorphically

class RobotDog:
    def speak(self):
        print("Beep boop — Woof!")

class RealDog:
    def speak(self):
        print("Woof!")

class Parrot:
    def speak(self):
        print("Polly wants a cracker!")

class Human:
    def speak(self):
        print("Hello, I can speak too!")


# A function that works with ANY object that has a speak() method
def make_it_speak(thing):
    thing.speak()    # Python just looks for speak() — doesn't care about the type


speakers = [RobotDog(), RealDog(), Parrot(), Human()]
for s in speakers:
    make_it_speak(s)

In [ ]:
# Duck typing also explains why len() works on str, list, dict, tuple, etc.
# They all implement __len__()

items = ["hello", [1, 2, 3], {"a": 1}, (4, 5)]
for item in items:
    print(f"{type(item).__name__:<10} len = {len(item)}")

---
## 9. 🚧 Abstract Classes

An **Abstract Class** is a class that:
- **Cannot be instantiated directly** — it's just a blueprint.
- **Forces subclasses** to implement certain methods.

Use `abc` module — **A**bstract **B**ase **C**lass.

```
  @abstractmethod → subclass MUST implement this method, or Python raises TypeError
```

This is the Pythonic way to implement **abstraction** — one of the 4 OOP pillars.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):           # inherit from ABC to make it abstract

    @abstractmethod
    def area(self):
        """Every shape MUST implement area()."""
        pass

    @abstractmethod
    def perimeter(self):
        """Every shape MUST implement perimeter()."""
        pass

    # Non-abstract method — shared by all subclasses, no override required
    def describe(self):
        print(f"{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}")


# ❌ Cannot instantiate abstract class directly
try:
    s = Shape()
except TypeError as e:
    print(f"TypeError: {e}")

In [ ]:
import math

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius


# ❌ Incomplete subclass — doesn't implement perimeter()
class BadShape(Shape):
    def area(self):
        return 0
    # forgot perimeter! ← Python will raise TypeError

try:
    bad = BadShape()
except TypeError as e:
    print(f"TypeError: {e}")

# ✅ Proper implementation works fine
c = Circle(7)
c.describe()

In [ ]:
# Real-world example: Payment system
from abc import ABC, abstractmethod

class PaymentProcessor(ABC):

    @abstractmethod
    def process_payment(self, amount):
        pass

    @abstractmethod
    def refund(self, amount):
        pass

    def receipt(self, amount):
        print(f"[{self.__class__.__name__}] Receipt issued for Rs. {amount}")


class CreditCardProcessor(PaymentProcessor):
    def process_payment(self, amount):
        print(f"Processing credit card payment: Rs. {amount}")
        self.receipt(amount)

    def refund(self, amount):
        print(f"Refunding to credit card: Rs. {amount}")


class EasyPaisaProcessor(PaymentProcessor):
    def process_payment(self, amount):
        print(f"Processing EasyPaisa payment: Rs. {amount}")
        self.receipt(amount)

    def refund(self, amount):
        print(f"Refunding via EasyPaisa: Rs. {amount}")


# Same interface — different implementations
processors = [CreditCardProcessor(), EasyPaisaProcessor()]
for p in processors:
    p.process_payment(5000)
    print()

---
## 10. ✨ Dunder (Magic) Methods

Dunder methods (double underscore: `__method__`) let you define how your objects behave with **Python's built-in operations** — operators, functions, and protocols.

### 10.1 Comparison Methods

In [ ]:
class Student:
    def __init__(self, name, gpa):
        self.name = name
        self.gpa  = gpa

    # ── Comparison dunders ────────────────────────────────
    def __eq__(self, other):   # ==
        return self.gpa == other.gpa

    def __lt__(self, other):   # <
        return self.gpa < other.gpa

    def __le__(self, other):   # <=
        return self.gpa <= other.gpa

    def __gt__(self, other):   # >
        return self.gpa > other.gpa

    def __ge__(self, other):   # >=
        return self.gpa >= other.gpa

    def __str__(self):
        return f"{self.name} (GPA: {self.gpa})"


s1 = Student("Ahmed", 3.8)
s2 = Student("Sara",  3.5)
s3 = Student("Ali",   3.8)

print(s1 > s2)     # True
print(s1 == s3)    # True  — same GPA
print(s2 < s1)     # True

# Now we can sort a list of students!
students = [s1, s2, s3, Student("Zara", 3.9)]
students.sort()    # uses __lt__ internally
for s in students:
    print(s)

### 10.2 Arithmetic Methods

In [ ]:
class Vector:
    """2D Mathematical Vector."""

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):      # v1 + v2
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):      # v1 - v2
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):     # v * 3  (scalar multiplication)
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):    # 3 * v  (reversed — scalar on left)
        return self.__mul__(scalar)

    def __neg__(self):             # -v
        return Vector(-self.x, -self.y)

    def __abs__(self):             # abs(v) — magnitude
        return (self.x**2 + self.y**2) ** 0.5

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __str__(self):
        return f"Vector({self.x}, {self.y})"

    def __repr__(self):
        return f"Vector(x={self.x}, y={self.y})"


v1 = Vector(3, 4)
v2 = Vector(1, 2)

print(v1 + v2)      # Vector(4, 6)
print(v1 - v2)      # Vector(2, 2)
print(v1 * 3)       # Vector(9, 12)
print(2 * v1)       # Vector(6, 8) — uses __rmul__
print(-v1)          # Vector(-3, -4)
print(abs(v1))      # 5.0  — magnitude of (3, 4) = 5
print(v1 == Vector(3, 4))   # True

### 10.3 Container Methods — `__len__`, `__getitem__`, `__contains__`

In [ ]:
class Playlist:
    """A music playlist that behaves like a sequence."""

    def __init__(self, name):
        self.name   = name
        self._songs = []

    def add_song(self, song):
        self._songs.append(song)

    def __len__(self):              # len(playlist)
        return len(self._songs)

    def __getitem__(self, index):   # playlist[0], playlist[1:3]
        return self._songs[index]

    def __contains__(self, song):   # 'song' in playlist
        return song in self._songs

    def __iter__(self):             # for song in playlist:
        return iter(self._songs)

    def __str__(self):
        return f"Playlist '{self.name}' ({len(self)} songs)"


pl = Playlist("My Favourites")
pl.add_song("Bohemian Rhapsody")
pl.add_song("Hotel California")
pl.add_song("Stairway to Heaven")
pl.add_song("Imagine")

print(pl)                              # uses __str__
print(f"Length: {len(pl)}")           # uses __len__
print(f"First song: {pl[0]}")         # uses __getitem__
print(f"Last song: {pl[-1]}")         # uses __getitem__
print(f"Slice: {pl[1:3]}")            # uses __getitem__ with slice

print("Hotel California" in pl)        # uses __contains__
print("Shape of You" in pl)            # False

print("\nAll songs:")
for song in pl:                        # uses __iter__
    print(f"  🎵 {song}")

### 10.4 Context Manager — `__enter__` and `__exit__`

The `with` statement uses these dunder methods. It guarantees **cleanup code always runs**, even if an error occurs.

In [ ]:
class DatabaseConnection:
    """Simulates a database connection as a context manager."""

    def __init__(self, db_name):
        self.db_name = db_name
        self.connected = False

    def __enter__(self):
        """Called when entering the 'with' block."""
        self.connected = True
        print(f"Connected to database: {self.db_name}")
        return self                  # returned as the 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Called when exiting the 'with' block — even if an error occurred."""
        self.connected = False
        print(f"Connection to {self.db_name} closed.")
        return False                 # False = don't suppress exceptions

    def query(self, sql):
        if not self.connected:
            raise RuntimeError("Not connected!")
        print(f"Running query: {sql}")
        return [{"id": 1, "name": "Ahmed"}, {"id": 2, "name": "Sara"}]


# with statement guarantees __exit__ is called
with DatabaseConnection("students_db") as db:
    results = db.query("SELECT * FROM students")
    for row in results:
        print(f"  {row}")

# After with block: connection is closed
print(f"Connected after block: {db.connected}")

### 10.5 Quick Reference — Common Dunder Methods

| Dunder Method | Triggered By | Purpose |
|---|---|---|
| `__init__` | `Obj()` | Constructor |
| `__str__` | `print(obj)`, `str(obj)` | Readable string |
| `__repr__` | `repr(obj)`, shell | Developer string |
| `__len__` | `len(obj)` | Length |
| `__getitem__` | `obj[i]` | Index access |
| `__setitem__` | `obj[i] = v` | Index assignment |
| `__contains__` | `x in obj` | Membership test |
| `__iter__` | `for x in obj` | Iteration |
| `__next__` | `next(obj)` | Iterator protocol |
| `__eq__` | `==` | Equality |
| `__lt__`, `__gt__` | `<`, `>` | Comparison |
| `__add__`, `__sub__` | `+`, `-` | Arithmetic |
| `__mul__`, `__truediv__` | `*`, `/` | Arithmetic |
| `__neg__`, `__abs__` | `-obj`, `abs(obj)` | Unary ops |
| `__bool__` | `bool(obj)`, `if obj:` | Truthiness |
| `__call__` | `obj()` | Make object callable |
| `__enter__`, `__exit__` | `with obj:` | Context manager |

In [ ]:
# Bonus: __bool__ and __call__

class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner   = owner
        self.balance = balance

    def __bool__(self):
        """Account is truthy if balance > 0."""
        return self.balance > 0

    def __call__(self, amount):
        """Calling the object directly deposits money."""
        self.balance += amount
        print(f"{self.owner}: Deposited {amount}. New balance: {self.balance}")

    def __str__(self):
        return f"Account({self.owner}, balance={self.balance})"


acc = BankAccount("Ahmed", 0)

# __bool__
if acc:
    print("Account has funds.")
else:
    print("Account is empty!")

# __call__ — calling the object like a function
acc(500)    # deposit 500
acc(300)    # deposit 300

if acc:
    print("Account has funds now.")

---
## 11. 🎯 Comprehensive Project — School Management System

Let's tie together **everything from Part 1 and Part 2** in a realistic mini-project.

In [ ]:
from abc import ABC, abstractmethod

# ── Abstract Base ─────────────────────────────────────────
class Person(ABC):
    """Abstract base — cannot be instantiated directly."""

    def __init__(self, name, age, email):
        self.name  = name
        self.age   = age
        self.email = email

    @abstractmethod
    def get_role(self):
        pass

    @abstractmethod
    def get_summary(self):
        pass

    def __str__(self):
        return f"[{self.get_role()}] {self.name}"

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, age={self.age})"


# ── Student class ─────────────────────────────────────────
class Student(Person):

    _student_count = 0

    def __init__(self, name, age, email, student_id):
        super().__init__(name, age, email)
        self.student_id = student_id
        self.__grades   = {}
        Student._student_count += 1

    def get_role(self):
        return "Student"

    def add_grade(self, subject, grade):
        if 0 <= grade <= 100:
            self.__grades[subject] = grade

    @property
    def gpa(self):
        if not self.__grades: return 0.0
        avg = sum(self.__grades.values()) / len(self.__grades)
        return round(avg / 25, 2)   # scale 0-100 to 0-4.0

    def get_summary(self):
        return f"  {self.name} | ID: {self.student_id} | GPA: {self.gpa} | Grades: {self.__grades}"

    @classmethod
    def total(cls):
        return cls._student_count

    # Comparison by GPA
    def __lt__(self, other):
        return self.gpa < other.gpa

    def __eq__(self, other):
        return self.student_id == other.student_id


# ── Teacher class ─────────────────────────────────────────
class Teacher(Person):

    def __init__(self, name, age, email, subject, salary):
        super().__init__(name, age, email)
        self.subject  = subject
        self.__salary = salary

    def get_role(self):
        return "Teacher"

    def get_summary(self):
        return f"  {self.name} | Subject: {self.subject} | Email: {self.email}"

    @property
    def salary(self):
        return self.__salary

    @salary.setter
    def salary(self, value):
        if value < 0:
            raise ValueError("Salary cannot be negative.")
        self.__salary = value


# ── School class (container with dunder methods) ──────────
class School:

    def __init__(self, name):
        self.name     = name
        self._members = []

    def enroll(self, person):
        if isinstance(person, Person):
            self._members.append(person)
            print(f"Enrolled: {person}")
        else:
            raise TypeError("Only Person subclasses can be enrolled.")

    def get_students(self):
        return [m for m in self._members if isinstance(m, Student)]

    def get_teachers(self):
        return [m for m in self._members if isinstance(m, Teacher)]

    def top_students(self, n=3):
        return sorted(self.get_students(), reverse=True)[:n]

    # Dunder methods
    def __len__(self):              # len(school)
        return len(self._members)

    def __contains__(self, person): # person in school
        return person in self._members

    def __iter__(self):             # for p in school:
        return iter(self._members)

    def __str__(self):
        return (f"🏫 {self.name} | "
                f"Students: {len(self.get_students())} | "
                f"Teachers: {len(self.get_teachers())}")


# ─────────── Demo ─────────────────────────────────────────
school = School("Python Academy")

# Enroll teachers
t1 = Teacher("Dr. Hassan",  45, "hassan@py.edu", "Mathematics", 120000)
t2 = Teacher("Ms. Ayesha",  35, "ayesha@py.edu", "Physics",     110000)
school.enroll(t1)
school.enroll(t2)

# Enroll students
s1 = Student("Ahmed Khan", 20, "ahmed@py.edu", "PY-001")
s2 = Student("Sara Ali",   21, "sara@py.edu",  "PY-002")
s3 = Student("Bilal Raza", 19, "bilal@py.edu", "PY-003")
s4 = Student("Zara Noor",  22, "zara@py.edu",  "PY-004")

for s in [s1, s2, s3, s4]:
    school.enroll(s)

# Add grades
s1.add_grade("Math", 88); s1.add_grade("Physics", 76); s1.add_grade("CS", 95)
s2.add_grade("Math", 92); s2.add_grade("Physics", 89); s2.add_grade("CS", 91)
s3.add_grade("Math", 65); s3.add_grade("Physics", 70); s3.add_grade("CS", 80)
s4.add_grade("Math", 98); s4.add_grade("Physics", 94); s4.add_grade("CS", 97)

print()
print(school)                    # __str__
print(f"Total members: {len(school)}")  # __len__
print(f"Ahmed enrolled: {s1 in school}") # __contains__

print("\n── All Members ──")
for member in school:            # __iter__
    print(member.get_summary())

print("\n── Top 3 Students ──")
for s in school.top_students(3): # sorted using __lt__
    print(f"  {s.name} — GPA: {s.gpa}")

print(f"\nTotal students ever enrolled: {Student.total()}")

---
## 12. 🧪 Practice Exercises

### Exercise 1 — Vehicle Hierarchy
Build the following hierarchy using inheritance:
```
        Vehicle  (abstract: fuel_type, speed, move())
       /       \
   Car          Motorcycle
    |                |
 ElectricCar    SportsBike
```
- `Vehicle`: abstract class with abstract method `move()` and `describe()`
- `Car`: adds `num_doors`, overrides `move()`
- `Motorcycle`: adds `has_sidecar`, overrides `move()`
- `ElectricCar(Car)`: overrides `fuel_type` to always be "Electric"
- `SportsBike(Motorcycle)`: adds a `turbo_boost()` method

In [ ]:
# ✏️ Your solution here
from abc import ABC, abstractmethod

class Vehicle(ABC):
    pass

### Exercise 2 — Custom List with Dunders
Create a `NumberList` class that wraps a list of numbers and supports:
- `+` to merge two NumberLists
- `*` to repeat the list (like `list * n`)
- `len()`, indexing `[]`, iteration `for x in`
- `in` for membership
- `bool()` — True if list is non-empty
- `__str__` showing the numbers and their sum
- A `stats()` method printing min, max, average

In [ ]:
# ✏️ Your solution here

class NumberList:
    pass

# Test:
# nl1 = NumberList([1, 2, 3, 4])
# nl2 = NumberList([5, 6, 7])
# print(nl1 + nl2)   # NumberList([1,2,3,4,5,6,7]) sum=28
# print(nl1 * 2)     # NumberList([1,2,3,4,1,2,3,4])
# print(3 in nl1)    # True
# nl1.stats()

### Exercise 3 — Mixin Pattern (Multiple Inheritance)
Create a logging mixin and a serialization mixin:
- `LogMixin`: adds `log(message)` that prints `[ClassName] message`
- `SerializeMixin`: adds `to_dict()` returning object's `__dict__` and `from_dict(data)` classmethod
- Create a `Product` class that inherits from both mixins and has `name`, `price`, `quantity`
- Test that logging and serialization work on `Product` instances

In [ ]:
# ✏️ Your solution here

class LogMixin:
    pass

class SerializeMixin:
    pass

class Product(LogMixin, SerializeMixin):
    pass

---
## 📋 Summary

| Concept | Key Points |
|---|---|
| **Inheritance** | `class Child(Parent)` — reuse & extend |
| **`super()`** | Access parent class — most common in `__init__` |
| **Method Overriding** | Child redefines a parent method |
| **`isinstance`** | Checks if object is instance of class (or subclass) |
| **`issubclass`** | Checks class-to-class relationship |
| **Multiple Inheritance** | `class Child(A, B)` — inherit from many parents |
| **MRO** | Python's left-to-right, depth-first method lookup order |
| **Polymorphism** | Same method name, different behavior per class |
| **Duck Typing** | Behavior > type — if it has the method, use it |
| **Abstract Classes** | `ABC` + `@abstractmethod` — enforce interface contracts |
| **Dunder Methods** | `__len__`, `__add__`, `__iter__` etc. — integrate with Python |

---

## ✅ OOP Topics Completed

```
Part 1 ✅  Classes, Objects, __init__, self
            Instance/Class/Static Methods
            __str__ / __repr__
            Encapsulation, @property

Part 2 ✅  Inheritance, super()
            Method Overriding
            Multiple Inheritance & MRO
            Polymorphism & Duck Typing
            Abstract Classes
            Dunder Methods
```

## ➡️ What's Next?

Suggested topics for future notebooks:
- **Modules & Packages** — organizing code into files
- **Exception Handling** — custom exceptions and class hierarchies
- **File I/O** — reading and writing files with context managers
- **Iterators & Generators** — building lazy sequences
- **Decorators** — function and class decorators in depth
- **Standard Library** — `collections`, `functools`, `itertools`

---
*Python Course | Object-Oriented Programming — Part 2*